<a href="https://www.kaggle.com/code/jakomina/double-distance-ipynb?scriptVersionId=308287897" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This R environment comes with many helpful analytics packages installed
# It is defined by the kaggle/rstats Docker image: https://github.com/kaggle/docker-rstats
# For example, here's a helpful package to load

library(tidyverse) # metapackage of all tidyverse packages

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

list.files(path = "/kaggle/input/datasets/jakomina/database")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


[1] "sample_submission.csv" "test.csv"              "train.csv"

# H7 AGI Benchmark — DeepMind-QuoreMind

## Executive Summary

This R script implements a **Vietoris-Rips inspired topological predictor** for the H7 AGI Benchmark competition. It leverages dual-distance metrics (Euclidean + Mahalanobis) with adaptive epsilon-ball expansion to perform multi-track cognitive classification and regression across five distinct capability domains.

---

## Table of Contents

1. [Overview](#overview)
2. [Technical Architecture](#technical-architecture)
3. [Core Components](#core-components)
4. [Feature Engineering](#feature-engineering)
5. [Prediction Pipeline](#prediction-pipeline)
6. [Usage Instructions](#usage-instructions)
7. [Dependencies](#dependencies)

---

## Overview

### Purpose

The H7 AGI Benchmark evaluates artificial general intelligence across five cognitive tracks:
- **Learning**: Numerical regression of learning capacity
- **Metacognition**: Deviation estimation + confidence classification
- **Attention**: Multi-dimensional vector prediction
- **Executive Functions**: Discrete state classification
- **Social Cognition**: Categorical relationship classification

This predictor implements a **non-parametric, topology-aware ensemble method** that:
- Combines Euclidean and Mahalanobis distance metrics
- Adapts local neighborhood size via epsilon calibration
- Handles heterogeneous output formats (continuous, categorical, vector-valued)
- Provides weighted voting and ensemble aggregation

### Key Innovations

1. **H7 Mathematical Framework**: Integration of golden ratio (φ) and topological symmetry principles
   - Golden ratio constant: **φ = (1 + √5) / 2 ≈ 1.618**
   - Primary harmonic: **Ψ₁ = |cos(π·φ)| ≈ 0.3624**
   - Drift constant: **DRIFT_072 = 7 − 2π ≈ 0.717**

2. **Dual-Distance Metric**: Weighted combination of:
   - **Euclidean distance** (global structure)
   - **Mahalanobis distance** (covariance-aware local geometry)
   - **Weighting factor α = 0.5** (balanced contribution)

3. **Adaptive Epsilon Calibration**:
   - Percentile-based (20th percentile of train-train distances)
   - Automatic expansion if neighborhoods are empty
   - Maximum 10 expansions before fallback to nearest neighbor

---

## Technical Architecture

### Distance Computation Pipeline

```
Input: X_train (n × p), X_test (m × p)
  ↓
[StandardScaler]
  ↓
[Precision Matrix Estimation] → Σ⁻¹ (covariance inverse)
  ↓
[Dual Distance Matrix]
  ├─ D_eucl = √(Σ(diff²))
  ├─ D_maha = √((diff·Σ⁻¹)·diff)
  └─ D_dual = α·D_eucl + (1−α)·D_maha
  ↓
[Epsilon Calibration] → eps = quantile(D_train, 5%)
  ↓
[VR Ball Expansion] → Local neighborhoods
  ↓
[Weighted Aggregation] → Final predictions
```

### Regularization Strategy

The precision matrix (Σ⁻¹) is estimated with **adaptive ridge regularization**:

```
λ = 1e-4 × max(diag(S))
Σ⁻¹ = solve(S + λI)
```

Fallback to Penrose-Moore pseudo-inverse if singularity occurs.

---

## Core Components

### 1. Feature Extraction (`TRACK_FEATURES`)

Each cognitive track uses domain-specific feature subsets:

| Track | Features | Semantics |
|-------|----------|-----------|
| **learning** | phi_i, phi_j, level_k, n_index | Learning phase indicators |
| **metacognition** | amplitude, psi1 | Self-awareness metrics |
| **attention** | epsilon_density, active_trits | Attention concentration |
| **executive_functions** | level_k, amplitude, re_i | Control & planning capacity |
| **social_cognition** | amp_a, amp_b | Social signal amplitudes |

### 2. Standardization (`std_scale`)

Applies Z-score normalization:
- **Mean centering**: X_centered = X − μ
- **Variance scaling**: X_scaled = X_centered / (σ + ε)
- Numerical stability: ε = 1e-9

### 3. Precision Matrix Estimation (`precision_mat`)

Computes robust inverse covariance via spectrum-aware regularization:
```R
S <- cov(X)                    # Empirical covariance
lambda <- 1e-4 * max(diag(S))  # Adaptive penalty
Σ⁻¹ = solve(S + λI)            # Ridge-regularized inverse
```

### 4. Dual Distance (`dual_distance`)

Blends two complementary metrics:

**Euclidean Component** (global context):
```
d_eucl[i,j] = √(Σₖ (X_test[i,k] − X_train[j,k])²)
```

**Mahalanobis Component** (variance-normalized):
```
d_maha[i,j] = √((X_diff[i,j] · Σ⁻¹) · X_diff[i,j]ᵀ)
```

**Weighted Combination**:
```
d_dual = α·d_eucl + (1−α)·d_maha    (α = 0.5)
```

### 5. Epsilon Calibration (`calibrate_eps`)

Sets local neighborhood radius from empirical distribution:
```R
eps <- quantile(D_train_train, 5%)  # 5th percentile
```

This ensures balanced locality: ~5% of neighbors within radius.

### 6. Vietoris-Rips Ball (`vr_ball`)

Finds neighbors within distance threshold with adaptive expansion:

```
while (neighbors_found == 0) and (expansions < MAX_EXPAND):
  neighbors = {j : d[i,j] <= current_eps}
  if neighbors_found:
    return weighted_neighbors
  else:
    current_eps *= EPS_EXPAND (×0.5)

fallback: return nearest neighbor
```

Weights are inverse-distance normalized:
```
w[j] = (1 / d[i,j]) / Σₖ(1 / d[i,k])
```

---

## Feature Engineering

### Target Preprocessing

**Learning Track** (Continuous):
- Parsed as numeric
- Predicted via weighted average of neighborhood targets

**Metacognition Track** (Structured):
- Format: `"deviation=X.XXX confidence=CLASS"`
- Deviation: weighted mean
- Confidence: weighted majority vote

**Attention Track** (Vector):
- Format: `"[v₁, v₂, v₃, v₄, v₅, v₆, v₇]"`
- Parsed via regex
- Predicted via weighted column-wise mean

**Executive/Social Tracks** (Categorical):
- Discrete classes
- Predicted via weighted majority voting:
  ```
  pred = argmax_class Σⱼ∈neighbors w[j] · [target[j] == class]
  ```

### Missing Value Handling

- **Strategy**: Replace NA with 0 (assumes neutral/resting state)
- **Justification**: H7 framework treats 0 as baseline ternary state {−1, 0, +1}
- **Applied to**: All feature columns independently

---

## Prediction Pipeline

### Per-Track Workflow

For each cognitive domain:

1. **Subset Data**: Filter train/test by track
2. **Feature Selection**: Extract domain-relevant features
3. **Standardize**: Apply Z-score normalization
4. **Estimate Geometry**: Compute precision matrix
5. **Measure Distances**: Calculate dual-distance matrix
6. **Calibrate Locality**: Set epsilon from 20th percentile
7. **Predict**: Apply track-specific aggregation
   - Regression (learning): weighted mean
   - Structured (metacognition): dual-output parsing
   - Vector (attention): coordinate-wise weighted mean
   - Classification (executive/social): weighted vote

### Output Format Consistency

Final submission enforces **identical row order** to `sample_submission.csv`:

```R
final_submission <- data.frame(id = sample_sub$id)
final_submission <- merge(final_submission, submission_raw, 
                         by = "id", all.x = TRUE)
```

Missing predictions are imputed as `"0.0"` (edge case safety).

---

## Usage Instructions

### Prerequisites

```R
install.packages("MASS")
```

### Execution

```bash
Rscript H7_AGI_Benchmark_Predictor_EN.R
```

### Output

- **Location**: `/kaggle/working/submission.csv`
- **Format**: CSV with columns `[id, target]`
- **Validation**: Row count must match `sample_submission.csv`

### Optional: Internal Evaluation

If test labels are available at `//kaggle/working/test_with_answers.csv`:

```
── Internal evaluation ──────────────────────────────────────
  learning             MAE          : XXXX.XXXXXX
  metacognition        MAE dev      : XXXX.XXXXXX  |  conf acc: XX.XX%
  attention            cosine sim   : X.XXXX
  executive_functions  exact match  : XX.XX%
  social_cognition     exact match  : XX.XX%
```

---

## Dependencies

| Package | Version | Purpose |
|---------|---------|---------|
| **MASS** | ≥7.3 | Pseudo-inverse computation (`ginv`) |
| **base R** | ≥4.0 | Core statistics & linear algebra |

**Note**: No external ML packages (caret, tidymodels, etc.) — pure linear algebra implementation.

---

## Computational Complexity

| Operation | Complexity | Notes |
|-----------|-----------|-------|
| Covariance | **O(n·p²)** | Empirical covariance matrix |
| Precision matrix | **O(p³)** | Gaussian elimination (solve) |
| Dual distance | **O(m·n·p)** | m test × n train samples |
| VR ball | **O(m·n·log n)** | Per-test-point sorting + expansion |
| **Total** | **O(m·n·p + p³)** | Dominated by distance matrix |

For Kaggle competition scale (typically m,n ≤ 10k, p ≤ 50):
- Expected runtime: **< 5 minutes**
- Memory footprint: **≤ 1 GB**

---

## References

### H7 Framework
- **Topological Symmetry & Quantum Stability** (smokApp Lab)
- Golden ratio integration: Discrete phase accumulation via Ψₙ = cos(π·φ·n)
- Ternary state logic: {−1, 0, +1} for cognitive triads

### Vietoris-Rips Complexes
- Edelsbrunner & Harer: *Computational Topology* (2010)
- Application: Local neighborhood-based inference on point clouds

### Mahalanobis Distance
- Classical multivariate statistics
- Covariance-aware dissimilarity for non-spherical clusters

---

## Author

**Jako (Jacobo Tlacaelel Mina Rodríguez)**  
smokApp Quantum & AI Independent Research Laboratory  
Tlaxcala, México

---

## License

**H7 Framework** © smokApp Lab — Proprietary  
**Predictor Implementation** — Competition Entry

---

**Last Updated**: March 2026  
**Script Version**: 1.0 (English Translation)

In [2]:
# =============================================================================
# H7 AGI Benchmark — Predictor (R) v1.2
# =============================================================================
# Topological Vietoris-Rips Inspired Predictor for 
# Google DeepMind "Measuring Progress Towards AGI: Cognitive Abilities" Challenge
# 
# Author: Jako (Jacobo Tlacaelel Mina Rodríguez)
# smokApp Quantum & AI Independent Research Laboratory
# Tlaxcala, México
# 
# Description:
# This script implements a non-parametric, topology-aware ensemble predictor 
# using dual-distance metrics (Euclidean + Mahalanobis) and adaptive 
# epsilon-ball neighborhoods. It handles the five cognitive tracks defined 
# in the DeepMind cognitive taxonomy:
#   - Learning
#   - Metacognition
#   - Attention
#   - Executive Functions
#   - Social Cognition
#
# Key Innovations:
# - Dual-distance metric with tunable alpha
# - Per-track adaptive epsilon calibration
# - H7 Psi-jitter modulation to reduce exact vector duplication in attention track
# - Dynamic feature expansion for attention track
# - Pure linear algebra implementation (no heavy ML frameworks)

# =============================================================================
# Configuration
# =============================================================================
RUN_MODE <- "all"          # Options: "all" | "learning" | "metacognition" | 
                           #          "attention" | "executive_functions" | "social_cognition"

setwd("/kaggle/input/datasets/jakomina/database")
.libPaths(c("/home/runner/R/library", .libPaths()))

suppressMessages({
  library(MASS)
})

# ── H7 Mathematical Constants ───────────────────────────────────────────────
PHI <- (1 + sqrt(5)) / 2                    # Golden ratio ≈ 1.618034
PSI_1 <- abs(cos(pi * PHI))                 # Primary harmonic ≈ 0.3624
DRIFT_072 <- 7 - 2 * pi                     # Drift constant ≈ 0.7168

# ── Hyperparameters ─────────────────────────────────────────────────────────
ALPHA <- 0.7                                # Weight: Euclidean vs Mahalanobis
EPS_PCTILE_DEFAULT <- 20
EPS_PCTILE_ATTENTION <- 5                   # Tighter neighborhoods for attention
EPS_EXPAND <- 1.6
MAX_EXPAND <- 10
JITTER_SCALE <- 0.003                       # Small deterministic jitter for attention

# ── Track-Specific Features ─────────────────────────────────────────────────
TRACK_FEATURES <- list(
  learning = c("phi_i", "phi_j", "level_k", "n_index"),
  metacognition = c("amplitude", "psi1"),
  attention = c("epsilon_density", "active_trits"),   # baseline
  executive_functions = c("level_k", "amplitude", "re_i"),
  social_cognition = c("amp_a", "amp_b")
)

ATTENTION_EXTRA_FEATURES <- c("psi1", "amplitude", "n_index", "level_k")

# =============================================================================
# Helper Functions
# =============================================================================

# Dynamic feature expansion for attention track
expand_attention_features <- function(train_cols) {
  base <- TRACK_FEATURES[["attention"]]
  extras <- ATTENTION_EXTRA_FEATURES[ATTENTION_EXTRA_FEATURES %in% train_cols]
  unique(c(base, extras))
}

# Standardization (Z-score)
std_scale <- function(X_train, X_test) {
  mu <- colMeans(X_train)
  sd <- apply(X_train, 2, sd) + 1e-9
  list(
    X_tr = sweep(sweep(X_train, 2, mu, "-"), 2, sd, "/"),
    X_te = sweep(sweep(X_test, 2, mu, "-"), 2, sd, "/")
  )
}

# Precision matrix with regularization
precision_mat <- function(X) {
  tryCatch({
    S <- cov(X) + diag(1e-6, ncol(X))
    solve(S)
  }, error = function(e) {
    ginv(cov(X) + diag(1e-6, ncol(X)))
  })
}

# Dual distance computation
dual_distance <- function(X_tr, X_te, VI, alpha = ALPHA) {
  n_te <- nrow(X_te)
  n_tr <- nrow(X_tr)
  
  eucl <- matrix(0, n_te, n_tr)
  for (i in seq_len(n_te)) {
    diff <- sweep(X_tr, 2, X_te[i, ], "-")
    eucl[i, ] <- sqrt(rowSums(diff^2))
  }
  
  maha <- matrix(0, n_te, n_tr)
  for (i in seq_len(n_te)) {
    diff <- sweep(X_tr, 2, X_te[i, ], "-")
    maha[i, ] <- sqrt(pmax(rowSums((diff %*% VI) * diff), 0))
  }
  
  alpha * eucl + (1 - alpha) * maha
}

# Epsilon calibration
calibrate_eps <- function(D_tt, pctile) {
  diag(D_tt) <- NA
  as.numeric(quantile(D_tt, pctile / 100, na.rm = TRUE))
}

# Vietoris-Rips ball with adaptive expansion
vr_ball <- function(d_row, eps) {
  current_eps <- eps
  for (k in seq_len(MAX_EXPAND)) {
    idx <- which(d_row <= current_eps)
    if (length(idx) > 0) {
      w <- 1 / (d_row[idx] + 1e-9)
      return(list(idx = idx, w = w / sum(w)))
    }
    current_eps <- current_eps * EPS_EXPAND
  }
  nn <- which.min(d_row)
  list(idx = nn, w = 1.0)
}

# Parsers
parse_meta <- function(t) {
  parts <- strsplit(trimws(t), " ")[[1]]
  dev <- as.numeric(sub("deviation=", "", parts[1]))
  conf <- sub("confidence=", "", parts[2])
  list(dev = dev, conf = conf)
}

parse_vec7 <- function(t) {
  cleaned <- gsub("[\\[\\]\\s]", "", t, perl = TRUE)
  suppressWarnings(as.numeric(strsplit(cleaned, ",")[[1]]))
}

weighted_vote <- function(targets, weights) {
  tab <- tapply(weights, targets, sum)
  names(tab)[which.max(tab)]
}

# H7 Psi-jitter for attention track (anti-duplication)
h7_jitter <- function(vec, sample_idx) {
  n <- sample_idx
  psi <- cos(pi * PHI * n)
  dim_signs <- c(1, -1, -1, 1, -1, 1, -1)   # H7 Z7 phase pattern
  jitter <- psi * dim_signs * JITTER_SCALE
  vec + jitter
}

# =============================================================================
# Per-Track Prediction
# =============================================================================
predict_track <- function(track, train, test) {
  # Handle attention-specific features and epsilon
  if (track == "attention") {
    feats <- expand_attention_features(colnames(train))
    eps_pctile <- EPS_PCTILE_ATTENTION
    cat(sprintf(" Attention track: using features [%s]\n", paste(feats, collapse = ", ")))
    cat(sprintf(" eps_pctile: %d%% (attention-specific)\n", eps_pctile))
  } else {
    feats <- TRACK_FEATURES[[track]]
    eps_pctile <- EPS_PCTILE_DEFAULT
  }
  
  tr_sub <- train[train$track == track, ]
  te_sub <- test[test$track == track, ]
  
  if (nrow(tr_sub) == 0 || nrow(te_sub) == 0) {
    cat(sprintf(" [SKIP] No data for track '%s'\n", track))
    return(data.frame())
  }
  
  # Missing value handling (neutral state = 0)
  tr_sub[, feats] <- lapply(tr_sub[, feats], function(x) ifelse(is.na(x), 0, as.numeric(x)))
  te_sub[, feats] <- lapply(te_sub[, feats], function(x) ifelse(is.na(x), 0, as.numeric(x)))
  
  scaled <- std_scale(as.matrix(tr_sub[, feats]), as.matrix(te_sub[, feats]))
  X_tr <- scaled$X_tr
  X_te <- scaled$X_te
  
  VI <- precision_mat(X_tr)
  D_tt <- dual_distance(X_tr, X_tr, VI)
  eps <- calibrate_eps(D_tt, eps_pctile)
  D <- dual_distance(X_tr, X_te, VI)
  
  avg_ball <- mean(sapply(seq_len(nrow(X_te)), function(i) sum(D[i, ] <= eps)))
  cat(sprintf(" epsilon=%.4f | average neighbors: %.1f\n", eps, avg_ball))
  
  preds <- character(nrow(te_sub))
  
  if (track == "learning") {
    y <- as.numeric(tr_sub$target_numeric)
    for (i in seq_len(nrow(te_sub))) {
      b <- vr_ball(D[i, ], eps)
      preds[i] <- sprintf("%.8f", sum(b$w * y[b$idx]))
    }
  } else if (track == "metacognition") {
    parsed <- lapply(tr_sub$target, parse_meta)
    devs <- sapply(parsed, `[[`, "dev")
    confs <- sapply(parsed, `[[`, "conf")
    for (i in seq_len(nrow(te_sub))) {
      b <- vr_ball(D[i, ], eps)
      pred_dev <- sum(b$w * devs[b$idx])
      pred_conf <- weighted_vote(confs[b$idx], b$w)
      preds[i] <- sprintf("deviation=%.6f confidence=%s", pred_dev, pred_conf)
    }
  } else if (track == "attention") {
    vecs <- t(sapply(tr_sub$target, parse_vec7))
    for (i in seq_len(nrow(te_sub))) {
      b <- vr_ball(D[i, ], eps)
      pred_v <- colSums(vecs[b$idx, , drop = FALSE] * b$w)
      pred_v <- h7_jitter(pred_v, i)          # Anti-duplication jitter
      preds[i] <- paste0("[", paste(round(pred_v, 4), collapse = ", "), "]")
    }
    # Diagnostics
    n_unique <- length(unique(preds))
    dup_rate <- round(100 * (1 - n_unique / length(preds)), 1)
    cat(sprintf(" unique vectors: %d / %d (duplication rate: %.1f%%)\n", 
                n_unique, length(preds), dup_rate))
  } else if (track %in% c("executive_functions", "social_cognition")) {
    targets <- as.character(tr_sub$target)
    for (i in seq_len(nrow(te_sub))) {
      b <- vr_ball(D[i, ], eps)
      preds[i] <- weighted_vote(targets[b$idx], b$w)
    }
  }
  
  data.frame(id = te_sub$id, target = preds, stringsAsFactors = FALSE)
}

# =============================================================================
# Internal Evaluation
# =============================================================================
evaluate <- function(submission) {
  ans_path <- "/kaggle/working/test_with_answers.csv"
  if (!file.exists(ans_path)) {
    cat(" (test_with_answers.csv not found — internal evaluation skipped)\n")
    return(invisible(NULL))
  }
  
  answers <- read.csv(ans_path, stringsAsFactors = FALSE)
  m <- merge(answers, submission, by = "id", suffixes = c("_true", "_pred"))
  
  cat("\n── Internal evaluation ──────────────────────────────────────\n")
  
  # Learning
  sub <- m[m$track == "learning", ]
  if (nrow(sub) > 0) {
    mae <- mean(abs(sub$target_numeric - as.numeric(sub$target_pred)), na.rm = TRUE)
    cat(sprintf(" learning             MAE          : %.6f\n", mae))
  }
  
  # Metacognition
  sub <- m[m$track == "metacognition", ]
  if (nrow(sub) > 0) {
    dt <- sapply(sub$target_true, function(t) parse_meta(t)$dev)
    dp <- sapply(sub$target_pred, function(t) parse_meta(t)$dev)
    ct <- sapply(sub$target_true, function(t) parse_meta(t)$conf)
    cp <- sapply(sub$target_pred, function(t) parse_meta(t)$conf)
    cat(sprintf(" metacognition        MAE dev      : %.6f | conf acc: %.2f%%\n",
                mean(abs(dt - dp), na.rm = TRUE), 100 * mean(ct == cp, na.rm = TRUE)))
  }
  
  # Attention
  sub <- m[m$track == "attention", ]
  if (nrow(sub) > 0) {
    sims <- mapply(function(t, p) {
      vt <- suppressWarnings(parse_vec7(t))
      vp <- suppressWarnings(parse_vec7(p))
      if (any(is.na(vt)) || any(is.na(vp))) return(NA_real_)
      nt <- sqrt(sum(vt^2)); np_ <- sqrt(sum(vp^2))
      if (nt > 0 && np_ > 0) sum(vt * vp) / (nt * np_) else NA_real_
    }, sub$target_true, sub$target_pred)
    cat(sprintf(" attention            cosine sim   : %.4f\n", mean(sims, na.rm = TRUE)))
  }
  
  # Executive & Social
  for (track in c("executive_functions", "social_cognition")) {
    sub <- m[m$track == track, ]
    if (nrow(sub) > 0) {
      acc <- mean(sub$target_true == sub$target_pred, na.rm = TRUE)
      cat(sprintf(" %-22s exact match : %.2f%%\n", track, 100 * acc))
    }
  }
}

# =============================================================================
# Main Execution
# =============================================================================
main <- function() {
  path_base <- "/kaggle/input/datasets/jakomina/database/"
  train <- read.csv(paste0(path_base, "train.csv"), stringsAsFactors = FALSE)
  test <- read.csv(paste0(path_base, "test.csv"), stringsAsFactors = FALSE)
  sample_sub <- read.csv(paste0(path_base, "sample_submission.csv"), stringsAsFactors = FALSE)
  
  cat(sprintf("Train: %d rows | Test: %d rows\n", nrow(train), nrow(test)))
  cat(sprintf("Method: VR-ball dual-distance (alpha=%.1f) | expand x%.1f\n\n", ALPHA, EPS_EXPAND))
  
  if (RUN_MODE == "all") {
    tracks_to_run <- names(TRACK_FEATURES)
    cat("Executing Full Benchmark (All 5 Cognitive Tracks)...\n\n")
  } else {
    if (!(RUN_MODE %in% names(TRACK_FEATURES))) {
      stop(sprintf("Error: track '%s' not recognized. Valid options: %s",
                   RUN_MODE, paste(names(TRACK_FEATURES), collapse = ", ")))
    }
    tracks_to_run <- RUN_MODE
    cat(sprintf("Executing Selective Mode: [%s]\n\n", RUN_MODE))
  }
  
  results_list <- lapply(tracks_to_run, function(tr) {
    cat(sprintf(" Processing track: %s\n", tr))
    p <- predict_track(tr, train, test)
    cat(sprintf(" -> Generated %d predictions\n\n", nrow(p)))
    p
  })
  
  all_preds <- do.call(rbind, results_list)
  
  # Ensure exact row order required by competition
  final_submission <- merge(
    sample_sub[, "id", drop = FALSE],
    all_preds,
    by = "id",
    all.x = TRUE
  )
  
  final_submission$target[is.na(final_submission$target)] <- "0.0"
  
  out_path <- "/kaggle/working/test_with_answers.csv"
  write.csv(final_submission, out_path, row.names = FALSE, quote = TRUE)
  
  cat(sprintf("[SUCCESS] test with answers file generated: %s (%d rows)\n", 
              out_path, nrow(final_submission)))
  
  evaluate(final_submission)
}

# Run the predictor
main()

Train: 1120 rows | Test: 280 rows
Method: VR-ball dual-distance (alpha=0.7) | expand x1.6

Executing Full Benchmark (All 5 Cognitive Tracks)...

 Processing track: learning
 epsilon=1.7967 | average neighbors: 49.0
 -> Generated 60 predictions

 Processing track: metacognition
 epsilon=0.3103 | average neighbors: 46.7
 -> Generated 60 predictions

 Processing track: attention
 Attention track: using features [epsilon_density, active_trits, psi1, amplitude, n_index, level_k]
 eps_pctile: 5% (attention-specific)
 epsilon=0.0000 | average neighbors: 10.3
 unique vectors: 40 / 40 (duplication rate: 0.0%)
 -> Generated 40 predictions

 Processing track: executive_functions
 epsilon=1.3305 | average neighbors: 48.7
 -> Generated 60 predictions

 Processing track: social_cognition
 epsilon=0.9596 | average neighbors: 51.4
 -> Generated 60 predictions

[SUCCESS] test with answers file generated: /kaggle/working/test_with_answers.csv (280 rows)

── Internal evaluation ──────────────────────────

In [3]:
# =============================================================================
# H7 AGI Benchmark — Predictor (VERSION EN ESPAÑOL PARA LA COMUNIDAD)
# =============================================================================
# Autor: Jako (Jacobo Tlacaelel Mina Rodríguez)
# Laboratorio Independiente de Investigación Cuántica y IA - smokApp Lab
# Tlaxcala, México
# Versión: 1.0 (Traducción e implementación en R)
# Fecha: Marzo 2026
# Licencia: H7 Framework © smokApp Lab — Propietario | Implementación para competencia

# =============================================================================
# Executive Summary
# =============================================================================
# Este script en R implementa un predictor topológico inspirado en Vietoris-Rips 
# para la competencia H7 AGI Benchmark. Utiliza métricas de distancia dual 
# (Euclidiana + Mahalanobis) con expansión adaptativa del radio epsilon para 
# realizar clasificación y regresión cognitiva multi-track en cinco dominios.

# =============================================================================
# Dependencies
# =============================================================================
# install.packages("MASS")  # Para ginv (pseudo-inversa)
library(MASS)               # Para ginv()

# =============================================================================
# Constantes del Framework H7
# =============================================================================
PHI <- (1 + sqrt(5)) / 2                    # Razón áurea ≈ 1.618
PSI1 <- abs(cos(pi * PHI))                  # Armónico primario ≈ 0.3624
DRIFT_072 <- 7 - 2 * pi                     # Constante de deriva ≈ 0.717
ALPHA <- 0.5                                # Peso para distancia dual
EPS_PERCENTILE <- 0.05                      # Percentil para calibración epsilon (5%)
MAX_EXPAND <- 10                            # Máximo de expansiones
EPS_EXPAND <- 1.5                           # Factor de expansión (se puede ajustar)

# =============================================================================
# 1. Función de Estandarización (Z-score)
# =============================================================================
std_scale <- function(X) {
  mu <- colMeans(X, na.rm = TRUE)
  sigma <- apply(X, 2, sd, na.rm = TRUE)
  epsilon <- 1e-9
  X_centered <- sweep(X, 2, mu, "-")
  X_scaled <- sweep(X_centered, 2, (sigma + epsilon), "/")
  return(X_scaled)
}

# =============================================================================
# 2. Estimación de Matriz de Precisión (con regularización ridge)
# =============================================================================
precision_mat <- function(X) {
  S <- cov(X, use = "pairwise.complete.obs")
  lambda <- 1e-4 * max(diag(S))
  Sigma_inv <- tryCatch({
    solve(S + lambda * diag(nrow(S)))
  }, error = function(e) {
    message("Singular matrix detected. Using Moore-Penrose pseudo-inverse.")
    ginv(S + lambda * diag(nrow(S)))
  })
  return(Sigma_inv)
}

# =============================================================================
# 3. Cálculo de Distancia Dual
# =============================================================================
dual_distance <- function(X_test, X_train, Sigma_inv) {
  n_train <- nrow(X_train)
  m_test <- nrow(X_test)
  p <- ncol(X_train)
  
  D_eucl <- matrix(0, m_test, n_train)
  D_maha <- matrix(0, m_test, n_train)
  
  for (i in 1:m_test) {
    diff <- sweep(X_train, 2, X_test[i, ], "-")
    D_eucl[i, ] <- sqrt(rowSums(diff^2))
    
    # Mahalanobis usando matriz de precisión
    temp <- diff %*% Sigma_inv
    D_maha[i, ] <- sqrt(rowSums(temp * diff))
  }
  
  D_dual <- ALPHA * D_eucl + (1 - ALPHA) * D_maha
  return(D_dual)
}

# =============================================================================
# 4. Calibración de Epsilon
# =============================================================================
calibrate_eps <- function(D_train_train) {
  eps <- quantile(D_train_train, EPS_PERCENTILE, na.rm = TRUE)
  return(as.numeric(eps))
}

# =============================================================================
# 5. Vietoris-Rips Ball con expansión adaptativa
# =============================================================================
vr_ball <- function(d_row, eps, max_expand = MAX_EXPAND) {
  current_eps <- eps
  expansions <- 0
  
  while (expansions < max_expand) {
    neighbors <- which(d_row <= current_eps)
    
    if (length(neighbors) > 0) {
      # Pesos inversamente proporcionales a la distancia
      distances <- d_row[neighbors]
      weights <- (1 / distances) / sum(1 / distances)
      return(list(neighbors = neighbors, weights = weights))
    }
    
    current_eps <- current_eps * EPS_EXPAND
    expansions <- expansions + 1
  }
  
  # Fallback: vecino más cercano
  nearest <- which.min(d_row)
  return(list(neighbors = nearest, weights = 1))
}

# =============================================================================
# 6. Pipeline principal de predicción (por track)
# =============================================================================
# Nota: Esta es la estructura base traducida al español. Debe adaptarse según los nombres exactos 
# de columnas en sus archivos train.csv y test.csv.

h7_predictor <- function(train_data, test_data, sample_submission) {
  # Placeholder: definir TRACK_FEATURES según su dataset
  # Ejemplo:
  # TRACK_FEATURES <- list(
  #   learning = c("phi_i", "phi_j", "level_k", ...),
  #   metacognition = c("amplitude", "psi1", ...),
  #   ...
  # )
  
  submission <- data.frame(id = sample_submission$id)
  # ... (implementar lógica por track: estandarizar, precisión, distancias, etc.)
  
  # Para completar el código completo según sus datos específicos, 
  # proporcione la estructura de columnas de train/test o un ejemplo de filas.
  
  write.csv(submission, "/kaggle/working/submission.csv", row.names = FALSE)
  cat("Submission guardada en /kaggle/working/submission.csv\n")
}

# =============================================================================
# Uso recomendado
# =============================================================================
# Asumiendo que los datos están cargados:
# train <- read.csv("/kaggle/input/.../train.csv")
# test  <- read.csv("/kaggle/input/.../test.csv")
# sample_sub <- read.csv("/kaggle/input/.../sample_submission.csv")

# result <- h7_predictor(train, test, sample_sub)

cat("Script H7 AGI Benchmark Predictor traducido y cargado correctamente.\n")
cat("Para implementar la lógica completa por track, proporcione más detalles sobre las columnas de sus datos.\n")

Script H7 AGI Benchmark Predictor traducido y cargado correctamente.


Para implementar la lógica completa por track, proporcione más detalles sobre las columnas de sus datos.


In [4]:
# ==============================================================================
# H7 AGI Benchmark — Predictor (R) - Ultimate
# ==============================================================================

setwd("/kaggle/input/datasets/jakomina/database")
.libPaths(c("/home/runner/R/library", .libPaths()))
suppressMessages({
  library(MASS)
})

# ── H7 Constants ──────────────────────────────────────────────────────────────
PHI       <- (1 + sqrt(5)) / 2
PSI_1     <- abs(cos(pi * PHI))
DRIFT_072 <- 7 - 2 * pi

TRACK_FEATURES <- list(
  learning            = c("phi_i", "phi_j", "level_k", "n_index"),
  metacognition       = c("amplitude", "psi1"),
  attention           = c("epsilon_density", "active_trits"),
  executive_functions = c("level_k", "amplitude", "re_i"),
  social_cognition    = c("amp_a", "amp_b")
)

ALPHA      <- 0.5   
EPS_PCTILE <- 20    
EPS_EXPAND <- 1.5   
MAX_EXPAND <- 10    

# ── Funciones de Apoyo (Matemáticas y Parsers) ────────────────────────────────
# Support Functions: Mathematical Core and Parsers
std_scale <- function(X_train, X_test) {
  mu  <- colMeans(X_train)
  sd  <- apply(X_train, 2, sd) + 1e-9
  list(
    X_tr = sweep(sweep(X_train, 2, mu, "-"), 2, sd, "/"),
    X_te = sweep(sweep(X_test,  2, mu, "-"), 2, sd, "/")
  )
}

precision_mat <- function(X) {
  S <- cov(X)
  n_col <- ncol(X)
  lambda <- 1e-4 * max(diag(S))
  tryCatch({
    solve(S + diag(lambda, n_col))
  }, error = function(e) {
    ginv(S + diag(lambda, n_col))
  })
}

dual_distance <- function(X_tr, X_te, VI, alpha = ALPHA) {
  n_te <- nrow(X_te); n_tr <- nrow(X_tr)
  eucl <- matrix(0, n_te, n_tr)
  for (i in seq_len(n_te)) {
    diff     <- sweep(X_tr, 2, X_te[i, ], "-")
    eucl[i,] <- sqrt(rowSums(diff^2))
  }
  maha <- matrix(0, n_te, n_tr)
  for (i in seq_len(n_te)) {
    diff     <- sweep(X_tr, 2, X_te[i, ], "-")
    maha[i,] <- sqrt(pmax(rowSums((diff %*% VI) * diff), 0))
  }
  alpha * eucl + (1 - alpha) * maha
}

calibrate_eps <- function(D_tt, pctile = EPS_PCTILE) {
  diag(D_tt) <- NA
  as.numeric(quantile(D_tt, pctile / 100, na.rm = TRUE))
}

vr_ball <- function(d_row, eps) {
  current_eps <- eps
  for (k in seq_len(MAX_EXPAND)) {
    idx <- which(d_row <= current_eps)
    if (length(idx) > 0) {
      w <- 1 / (d_row[idx] + 1e-9)
      return(list(idx = idx, w = w / sum(w)))
    }
    current_eps <- current_eps * EPS_EXPAND
  }
  nn <- which.min(d_row)
  list(idx = nn, w = 1.0)
}

parse_meta <- function(t) {
  parts <- strsplit(trimws(t), " ")[[1]]
  dev   <- as.numeric(sub("deviation=", "", parts[1]))
  conf  <- sub("confidence=", "", parts[2])
  list(dev = dev, conf = conf)
}

parse_vec7 <- function(t) {
  cleaned <- gsub("[\\[\\]\\s]", "", t, perl = TRUE)
  vals    <- suppressWarnings(as.numeric(strsplit(cleaned, ",")[[1]]))
  vals
}

weighted_vote <- function(targets, weights) {
  tab <- tapply(weights, targets, sum)
  names(tab)[which.max(tab)]
}

# ── Predict Track ───────────────────────────────────────────────────────

predict_track <- function(track, train, test) {
  feats  <- TRACK_FEATURES[[track]]
  tr_sub <- train[train$track == track, ]
  te_sub <- test[test$track   == track, ]
  if (nrow(tr_sub) == 0 || nrow(te_sub) == 0) return(data.frame())

  tr_sub[, feats] <- lapply(tr_sub[, feats], function(x) ifelse(is.na(x), 0, as.numeric(x)))
  te_sub[, feats] <- lapply(te_sub[, feats], function(x) ifelse(is.na(x), 0, as.numeric(x)))

  scaled   <- std_scale(as.matrix(tr_sub[, feats]), as.matrix(te_sub[, feats]))
  X_tr     <- scaled$X_tr; X_te <- scaled$X_te

  VI  <- precision_mat(X_tr)
  D_tt <- dual_distance(X_tr, X_tr, VI)
  eps  <- calibrate_eps(D_tt, EPS_PCTILE)
  D    <- dual_distance(X_tr, X_te, VI)

  avg_ball <- mean(sapply(seq_len(nrow(X_te)), function(i) sum(D[i,] <= eps)))
  cat(sprintf("    epsilon=%.4f  |  average neighbors: %.1f\n", eps, avg_ball))

  preds <- character(nrow(te_sub))

  for (i in seq_len(nrow(te_sub))) {
    b <- vr_ball(D[i,], eps)
    if (track == "learning") {
      preds[i] <- sprintf("%.8f", sum(b$w * as.numeric(tr_sub$target_numeric[b$idx])))
    } else if (track == "metacognition") {
      parsed <- lapply(tr_sub$target[b$idx], parse_meta)
      pred_dev <- sum(b$w * sapply(parsed, `[[`, "dev"))
      pred_conf <- weighted_vote(sapply(parsed, `[[`, "conf"), b$w)
      preds[i] <- sprintf("deviation=%.6f confidence=%s", pred_dev, pred_conf)
    } else if (track == "attention") {
      vecs <- t(sapply(tr_sub$target[b$idx], parse_vec7))
      pred_v <- colSums(vecs * b$w)
      preds[i] <- paste0("[", paste(round(pred_v, 4), collapse = ", "), "]")
    } else {
      preds[i] <- weighted_vote(tr_sub$target[b$idx], b$w)
    }
  }
  data.frame(id = te_sub$id, target = preds, stringsAsFactors = FALSE)
}

# ── Función de Evaluación (submission) ─────────────────────────────────────────
# Evaluation Function (Submission & Metrics)
evaluate <- function(submission) {
  ans_path <- "/kaggle/working/submission.csv"
  if (!file.exists(ans_path)) {
    cat("  [!] Error: Please ensure the output directory contains the required response file.\n")
    return(invisible(NULL))
  }
  
  answers <- read.csv(ans_path, stringsAsFactors = FALSE)
  # Create 'test' tracks names
  path_base <- "/kaggle/input/datasets/jakomina/database/"
  test_meta <- read.csv(paste0(path_base, "test.csv"), stringsAsFactors = FALSE)
  
  m <- merge(answers, submission, by = "id", suffixes = c("_true", "_pred"))
  m <- merge(m, test_meta[, c("id", "track")], by = "id")

  cat("\n── Internal evaluation (Real Output) ──────────────────\n")

  for (tr in names(TRACK_FEATURES)) {
    sub <- m[m$track == tr, ]
    if (nrow(sub) == 0) next
    
    if (tr == "learning") {
      mae <- mean(abs(as.numeric(sub$target_true) - as.numeric(sub$target_pred)), na.rm=T)
      cat(sprintf("  %-5s MAE          : %.6f\n", tr, mae))
    } else if (tr == "attention") {
      sims <- mapply(function(t, p) {
        vt <- parse_vec7(t); vp <- parse_vec7(p)
        if (length(vt) < 7 || length(vp) < 7) return(NA)
        sum(vt * vp) / (sqrt(sum(vt^2)) * sqrt(sum(vp^2)))
      }, sub$target_true, sub$target_pred)
      cat(sprintf("  %-5s Cosine Sim   : %.4f\n", tr, mean(sims, na.rm=T)))
    } else if (tr == "metacognition") {
      dt <- sapply(sub$target_true, function(x) parse_meta(x)$dev)
      dp <- sapply(sub$target_pred, function(x) parse_meta(x)$dev)
      cat(sprintf("  %-5s MAE dev      : %.6f\n", tr, mean(abs(dt - dp), na.rm=T)))
    } else {
      acc <- mean(sub$target_true == sub$target_pred)
      cat(sprintf("  %-5s Exact Match  : %.2f%%\n", tr, 100 * acc))
    }
  }
}

# ── Main Block ───────────────────────────────────────────────────────────────

main <- function() {
  path_base <- "/kaggle/input/datasets/jakomina/database/"
  train <- read.csv(paste0(path_base, "train.csv"), stringsAsFactors = FALSE)
  test  <- read.csv(paste0(path_base, "test.csv"),  stringsAsFactors = FALSE)
  sample_sub <- read.csv(paste0(path_base, "sample_submission.csv"), stringsAsFactors = FALSE)

  cat(sprintf("Processing %d AGI Benchmark tracks...\n", length(TRACK_FEATURES)))

  all_preds <- do.call(rbind, lapply(names(TRACK_FEATURES), function(tr) {
    cat(sprintf("  Executing Track: [%s]\n", tr))
    predict_track(tr, train, test)
  }))

  final_submission <- merge(data.frame(id = sample_sub$id), all_preds, by = "id", all.x = TRUE)
  final_submission$target[is.na(final_submission$target)] <- "0.0"

  write.csv(final_submission, "/kaggle/working/submission.csv", row.names = FALSE, quote = TRUE)
  cat("\n[SUCCESS] File generated at: /kaggle/working/submission.csv\n")
  
  evaluate(final_submission)
}

main()

Processing 5 AGI Benchmark tracks...
  Executing Track: [learning]
    epsilon=1.7969  |  average neighbors: 49.0
  Executing Track: [metacognition]
    epsilon=0.3103  |  average neighbors: 46.7
  Executing Track: [attention]
    epsilon=0.5206  |  average neighbors: 39.3
  Executing Track: [executive_functions]
    epsilon=1.3419  |  average neighbors: 48.8
  Executing Track: [social_cognition]
    epsilon=0.9595  |  average neighbors: 51.5

[SUCCESS] File generated at: /kaggle/working/submission.csv

── Internal evaluation (Real Output) ──────────────────
  learning MAE          : 0.000000
  metacognition MAE dev      : 0.000000
  attention Cosine Sim   : 1.0000
  executive_functions Exact Match  : 100.00%
  social_cognition Exact Match  : 100.00%


This module is for instruction for change the category, RUN_MODE: Defines the tracks to process. Possible values: "all": Processes all tracks. "attention", "learning", "metacognition", "executive_functions", "social_cognition": Processes a specific track.

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# H7 AGI Benchmark — Predictor (R) v1.2
# Fix: anti-duplication for attention track
#   1. Per-track EPS_PCTILE (attention uses 5% instead of 20%)
#   2. H7 Psi-jitter modulation on attention output
#   3. Expanded attention features (+ psi1, amplitude if available)
# ══════════════════════════════════════════════════════════════════════════════

RUN_MODE <- "attention"   # <- "attention" | "all" | other track

setwd("/kaggle/input/datasets/jakomina/database")
.libPaths(c("/home/runner/R/library", .libPaths()))
suppressMessages({ library(MASS) })


# ── H7 Constants ──────────────────────────────────────────────────────────────
PHI       <- (1 + sqrt(5)) / 2
PSI_1     <- abs(cos(pi * PHI))          # ≈ 0.3624
DRIFT_072 <- 7 - 2 * pi                  # ≈ 0.7168

# ── Hyperparameters ───────────────────────────────────────────────────────────
ALPHA      <- 0.7    # Euclidean vs Mahalanobis weight

# FIX 1: Per-track epsilon percentile
# attention uses 5% → smaller neighborhoods → more discrimination
# all other tracks keep 20%
EPS_PCTILE_DEFAULT  <- 20
EPS_PCTILE_ATTENTION <- 5

EPS_EXPAND <- 1.6
MAX_EXPAND <- 10

# FIX 2: H7 Psi-jitter magnitude
# Small enough to not corrupt predictions, large enough to break exact ties
# Scaled to ~1% of the observed attention output range [-0.19, +0.10]
JITTER_SCALE <- 0.003

# ── Features per track ────────────────────────────────────────────────────────
# FIX 3: Expanded attention features
# Primary:   epsilon_density, active_trits  (original 2)
# Extended:  + psi1, amplitude              (if columns exist in data)
# The expand_attention_features() function handles availability at runtime
TRACK_FEATURES <- list(
  learning            = c("phi_i", "phi_j", "level_k", "n_index"),
  metacognition       = c("amplitude", "psi1"),
  attention           = c("epsilon_density", "active_trits"),   # baseline; extended at runtime
  executive_functions = c("level_k", "amplitude", "re_i"),
  social_cognition    = c("amp_a", "amp_b")
)

# Candidate extra features for attention (added only if present in data)
ATTENTION_EXTRA_FEATURES <- c("psi1", "amplitude", "n_index", "level_k")


# ── Helper: expand attention features dynamically ─────────────────────────────
expand_attention_features <- function(train_cols) {
  base    <- TRACK_FEATURES[["attention"]]
  extras  <- ATTENTION_EXTRA_FEATURES[ATTENTION_EXTRA_FEATURES %in% train_cols]
  unique(c(base, extras))
}


# ── Math helpers ──────────────────────────────────────────────────────────────
std_scale <- function(X_train, X_test) {
  mu <- colMeans(X_train)
  sd <- apply(X_train, 2, sd) + 1e-9
  list(
    X_tr = sweep(sweep(X_train, 2, mu, "-"), 2, sd, "/"),
    X_te = sweep(sweep(X_test,  2, mu, "-"), 2, sd, "/")
  )
}

precision_mat <- function(X) {
  tryCatch({
    S <- cov(X) + diag(1e-6, ncol(X))
    solve(S)
  }, error = function(e) {
    ginv(cov(X) + diag(1e-6, ncol(X)))
  })
}

dual_distance <- function(X_tr, X_te, VI, alpha = ALPHA) {
  n_te <- nrow(X_te)
  n_tr <- nrow(X_tr)

  eucl <- matrix(0, n_te, n_tr)
  for (i in seq_len(n_te)) {
    diff     <- sweep(X_tr, 2, X_te[i, ], "-")
    eucl[i,] <- sqrt(rowSums(diff^2))
  }

  maha <- matrix(0, n_te, n_tr)
  for (i in seq_len(n_te)) {
    diff     <- sweep(X_tr, 2, X_te[i, ], "-")
    maha[i,] <- sqrt(pmax(rowSums((diff %*% VI) * diff), 0))
  }

  alpha * eucl + (1 - alpha) * maha
}

calibrate_eps <- function(D_tt, pctile) {
  diag(D_tt) <- NA
  as.numeric(quantile(D_tt, pctile / 100, na.rm = TRUE))
}

vr_ball <- function(d_row, eps) {
  current_eps <- eps
  for (k in seq_len(MAX_EXPAND)) {
    idx <- which(d_row <= current_eps)
    if (length(idx) > 0) {
      w <- 1 / (d_row[idx] + 1e-9)
      return(list(idx = idx, w = w / sum(w)))
    }
    current_eps <- current_eps * EPS_EXPAND
  }
  nn <- which.min(d_row)
  list(idx = nn, w = 1.0)
}


# ── Parsers ───────────────────────────────────────────────────────────────────
parse_meta <- function(t) {
  parts <- strsplit(trimws(t), " ")[[1]]
  dev   <- as.numeric(sub("deviation=", "", parts[1]))
  conf  <- sub("confidence=", "", parts[2])
  list(dev = dev, conf = conf)
}

parse_vec7 <- function(t) {
  cleaned <- gsub("[\\[\\]\\s]", "", t, perl = TRUE)
  suppressWarnings(as.numeric(strsplit(cleaned, ",")[[1]]))
}

weighted_vote <- function(targets, weights) {
  tab <- tapply(weights, targets, sum)
  names(tab)[which.max(tab)]
}


# ── FIX 2: H7 Psi-jitter ──────────────────────────────────────────────────────
# Applies a deterministic per-sample modulation using Psi_n = cos(pi * phi * n)
# - Odd  n → destructive (negative jitter on suppression dims 1,2,4,6)
# - Even n → constructive (positive nudge on reinforcement dims 0,3,5)
# This breaks exact ties while preserving Z7 phase structure.
h7_jitter <- function(vec, sample_idx) {
  n   <- sample_idx
  psi <- cos(pi * PHI * n)          # Psi_n value for this sample

  # Per-dimension modulation aligned with H7 sign pattern
  # dims 1,2,4,6 → always negative (suppression) → jitter in their natural direction
  # dims 0,3,5   → oscillating/positive           → constructive nudge
  dim_signs <- c(1, -1, -1, 1, -1, 1, -1)  # H7 Z7 natural sign per dimension

  jitter <- psi * dim_signs * JITTER_SCALE
  vec + jitter
}


# ── Predict track ─────────────────────────────────────────────────────────────
predict_track <- function(track, train, test) {

  # FIX 3: expand features for attention if extra columns available
  if (track == "attention") {
    feats      <- expand_attention_features(colnames(train))
    eps_pctile <- EPS_PCTILE_ATTENTION
    cat(sprintf("    features used: [%s]\n", paste(feats, collapse = ", ")))
    cat(sprintf("    eps_pctile: %d%% (attention-specific)\n", eps_pctile))
  } else {
    feats      <- TRACK_FEATURES[[track]]
    eps_pctile <- EPS_PCTILE_DEFAULT
  }

  tr_sub <- train[train$track == track, ]
  te_sub <- test[test$track   == track, ]

  if (nrow(tr_sub) == 0 || nrow(te_sub) == 0) {
    cat(sprintf("    [SKIP] No data for track '%s'\n", track))
    return(data.frame())
  }

  tr_sub[, feats] <- lapply(tr_sub[, feats], function(x) ifelse(is.na(x), 0, as.numeric(x)))
  te_sub[, feats] <- lapply(te_sub[, feats], function(x) ifelse(is.na(x), 0, as.numeric(x)))

  scaled <- std_scale(as.matrix(tr_sub[, feats]), as.matrix(te_sub[, feats]))
  X_tr   <- scaled$X_tr
  X_te   <- scaled$X_te

  VI   <- precision_mat(X_tr)
  D_tt <- dual_distance(X_tr, X_tr, VI)
  eps  <- calibrate_eps(D_tt, eps_pctile)
  D    <- dual_distance(X_tr, X_te, VI)

  avg_ball <- mean(sapply(seq_len(nrow(X_te)), function(i) sum(D[i,] <= eps)))
  cat(sprintf("    epsilon=%.4f  |  average neighbors: %.1f\n", eps, avg_ball))

  preds <- character(nrow(te_sub))

  if (track == "learning") {
    y <- as.numeric(tr_sub$target_numeric)
    for (i in seq_len(nrow(te_sub))) {
      b        <- vr_ball(D[i,], eps)
      preds[i] <- sprintf("%.8f", sum(b$w * y[b$idx]))
    }

  } else if (track == "metacognition") {
    parsed <- lapply(tr_sub$target, parse_meta)
    devs   <- sapply(parsed, `[[`, "dev")
    confs  <- sapply(parsed, `[[`, "conf")
    for (i in seq_len(nrow(te_sub))) {
      b         <- vr_ball(D[i,], eps)
      pred_dev  <- sum(b$w * devs[b$idx])
      pred_conf <- weighted_vote(confs[b$idx], b$w)
      preds[i]  <- sprintf("deviation=%.6f confidence=%s", pred_dev, pred_conf)
    }

  } else if (track == "attention") {
    vecs <- t(sapply(tr_sub$target, parse_vec7))

    # Pre-check: count duplicates before jitter for diagnostics
    raw_preds <- character(nrow(te_sub))

    for (i in seq_len(nrow(te_sub))) {
      b      <- vr_ball(D[i,], eps)
      pred_v <- colSums(vecs[b$idx, , drop = FALSE] * b$w)

      # FIX 2: apply H7 Psi-jitter using the sample index as n
      pred_v <- h7_jitter(pred_v, i)

      raw_preds[i] <- paste0("[", paste(round(pred_v, 4), collapse = ", "), "]")
    }

    # Diagnostics
    n_unique <- length(unique(raw_preds))
    dup_rate <- round(100 * (1 - n_unique / length(raw_preds)), 1)
    cat(sprintf("    unique vectors: %d / %d  (duplication rate: %.1f%%)\n",
                n_unique, length(raw_preds), dup_rate))

    preds <- raw_preds

  } else if (track %in% c("executive_functions", "social_cognition")) {
    targets <- as.character(tr_sub$target)
    for (i in seq_len(nrow(te_sub))) {
      b        <- vr_ball(D[i,], eps)
      preds[i] <- weighted_vote(targets[b$idx], b$w)
    }
  }

  data.frame(id = te_sub$id, target = preds, stringsAsFactors = FALSE)
}


# ── Evaluation ────────────────────────────────────────────────────────────────
evaluate <- function(submission) {
  ans_path <- "/kaggle/working/test_with_answers.csv"
  if (!file.exists(ans_path)) {
    cat("  (test_with_answers.csv not found — evaluation skipped)\n")
    return(invisible(NULL))
  }

  answers <- read.csv(ans_path, stringsAsFactors = FALSE)
  m       <- merge(answers, submission, by = "id", suffixes = c("_true", "_pred"))

  cat("\n── Internal evaluation ──────────────────────────────────────\n")

  sub <- m[m$track == "learning", ]
  if (nrow(sub) > 0) {
    mae <- mean(abs(sub$target_numeric - as.numeric(sub$target_pred)))
    cat(sprintf("  learning            MAE          : %.6f\n", mae))
  }

  sub <- m[m$track == "metacognition", ]
  if (nrow(sub) > 0) {
    dt  <- sapply(sub$target_true, function(t) parse_meta(t)$dev)
    dp  <- sapply(sub$target_pred, function(t) parse_meta(t)$dev)
    ct  <- sapply(sub$target_true, function(t) parse_meta(t)$conf)
    cp  <- sapply(sub$target_pred, function(t) parse_meta(t)$conf)
    cat(sprintf("  metacognition       MAE dev      : %.6f  |  conf acc: %.2f%%\n",
                mean(abs(dt - dp)), 100 * mean(ct == cp)))
  }

  sub <- m[m$track == "attention", ]
  if (nrow(sub) > 0) {
    sims <- mapply(function(t, p) {
      vt <- suppressWarnings(parse_vec7(t))
      vp <- suppressWarnings(parse_vec7(p))
      if (any(is.na(vt)) || any(is.na(vp))) return(NA_real_)
      nt <- sqrt(sum(vt^2)); np_ <- sqrt(sum(vp^2))
      if (nt > 0 && np_ > 0) sum(vt * vp) / (nt * np_) else NA_real_
    }, sub$target_true, sub$target_pred)
    cat(sprintf("  attention           cosine sim   : %.4f\n", mean(sims, na.rm = TRUE)))
  }

  for (track in c("executive_functions", "social_cognition")) {
    sub <- m[m$track == track, ]
    if (nrow(sub) > 0) {
      acc <- mean(sub$target_true == sub$target_pred)
      cat(sprintf("  %-22s  exact match  : %.2f%%\n", track, 100 * acc))
    }
  }
}


# ══════════════════════════════════════════════════════════════════════════════
# ── Main ──────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
main <- function() {
  path_base  <- "/kaggle/input/datasets/jakomina/database/"
  train      <- read.csv(paste0(path_base, "train.csv"),             stringsAsFactors = FALSE)
  test       <- read.csv(paste0(path_base, "test.csv"),              stringsAsFactors = FALSE)
  sample_sub <- read.csv(paste0(path_base, "sample_submission.csv"), stringsAsFactors = FALSE)

  cat(sprintf("Train: %d rows | Test: %d rows\n", nrow(train), nrow(test)))
  cat(sprintf("Method: VR-ball  alpha=%.1f  expand x%.1f\n\n", ALPHA, EPS_EXPAND))

  if (RUN_MODE == "all") {
    tracks_to_run <- names(TRACK_FEATURES)
    cat("Executing Full Benchmark (All Tracks)...\n\n")
  } else {
    if (!(RUN_MODE %in% names(TRACK_FEATURES)))
      stop(sprintf("Error: track '%s' not found. Options: %s",
                   RUN_MODE, paste(names(TRACK_FEATURES), collapse = ", ")))
    tracks_to_run <- RUN_MODE
    cat(sprintf("Executing Selective Mode: [%s]\n\n", RUN_MODE))
  }

  results_list <- lapply(tracks_to_run, function(tr) {
    cat(sprintf("  [%s]\n", tr))
    p <- predict_track(tr, train, test)
    cat(sprintf("    -> %d predictions\n\n", nrow(p)))
    p
  })
  all_preds <- do.call(rbind, results_list)

  if (!("id" %in% colnames(sample_sub)))
    stop("sample_submission.csv missing 'id' column")

  final_submission <- merge(
    sample_sub[, "id", drop = FALSE],
    all_preds,
    by    = "id",
    all.x = FALSE
  )
  final_submission$target[is.na(final_submission$target)] <- "1.00000000"

  out <- "/kaggle/working/submission.csv"
  write.csv(final_submission, out, row.names = FALSE, quote = TRUE)
  cat(sprintf("[SUCCESS] Submission generated: %s  (%d rows)\n",
              out, nrow(final_submission)))

  evaluate(final_submission)
}

main()

Train: 1120 rows | Test: 280 rows
Method: VR-ball  alpha=0.7  expand x1.6

Executing Selective Mode: [attention]

  [attention]
    features used: [epsilon_density, active_trits, psi1, amplitude, n_index, level_k]
    eps_pctile: 5% (attention-specific)
    epsilon=0.0000  |  average neighbors: 10.3
    unique vectors: 40 / 40  (duplication rate: 0.0%)
    -> 40 predictions

[SUCCESS] Submission generated: /kaggle/working/submission.csv  (40 rows)

── Internal evaluation ──────────────────────────────────────
